# HO3D Query Pipeline — `run_ho3d_query.py`

Runs and visualizes the query pose estimation step of Any6D on the HO3D dataset.
Reproduces the main results from the CVPR 2025 paper.

**Results folder structure:**
```
results/ho3d_results/any6d/<date>/
├── 0_mean_all_metrics_classes_results.xlsx
├── 0_all_frames_metrics_results.xlsx
└── <sequence>_metrics_results.xlsx
```

**Prerequisites:**
- `source ~/open-vocabulary-6d-pose-yoloe/master_env/bin/activate`
- Anchor poses computed (`04_ho3d_anchor_pipeline.ipynb` completed)
- HO3D evaluation dataset downloaded (~30-50 GB)
- YCB Video Models downloaded
- Docker running with the `any6d` container
- Kernel = `master_env`

**Expected time: ~3-5 hours for all 13 sequences**

## Cell 1 — Config & Imports

In [ ]:
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import sys

try:
    import openpyxl
except ImportError:
    subprocess.run(['pip', 'install', 'openpyxl', '-q'])
    import openpyxl

BASE_DIR     = os.path.expanduser('~/open-vocabulary-6d-pose-yoloe')
ANY6D_DIR    = os.path.join(BASE_DIR, 'Any6D')
ANCHOR_DIR   = os.path.join(ANY6D_DIR, 'anchor_results', 'dexycb_reference_view_ours')
HO3D_DIR     = '/home/josue_aims_ac_za/ssd_4tb/dataset/ho3d'
RESULTS_ROOT = os.path.join(ANY6D_DIR, 'results', 'ho3d_results')
VIZ_DIR      = os.path.join(BASE_DIR, 'notebooks', 'outputs')
os.makedirs(VIZ_DIR, exist_ok=True)

OBJECTS = [
    '006_mustard_bottle', '021_bleach_cleanser', '019_pitcher_base',
    '004_sugar_box', '005_tomato_soup_can', '003_cracker_box', '010_potted_meat_can'
]

HO3D_SEQUENCES = [
    'MPM10','MPM11','MPM12','MPM13','MPM14',
    'AP10','AP11','AP12','AP13','AP14',
    'SB11','SB13','SM1'
]

METRICS = ['ADD-S', 'ADD', 'AR', 'MSSD', 'MSPD', 'VSD']

sys.path.insert(0, os.path.join(BASE_DIR, 'utils'))
from any6d_utils import (
    docker_run, check_docker, check_docker_volumes,
    save_fig
)

print(f'ANY6D_DIR    : {ANY6D_DIR}')
print(f'HO3D_DIR     : {HO3D_DIR}')
print(f'RESULTS_ROOT : {RESULTS_ROOT}')

## Cell 2 — Check All Prerequisites

In [ ]:
all_ok = True
print('=== CHECKING PREREQUISITES ===')

print('\n[1] Anchor poses:')
for obj in OBJECTS:
    pose_path = os.path.join(ANCHOR_DIR, obj, f'{obj}_initial_pose.txt')
    if os.path.exists(pose_path):
        pose = np.loadtxt(pose_path)
        print(f'  OK  {obj:<30} Z={pose[2,3]*100:.1f}cm')
    else:
        print(f'  MISSING  {obj}  — run 04_ho3d_anchor_pipeline first')
        all_ok = False

print('\n[2] HO3D evaluation dataset:')
ho3d_eval = os.path.join(HO3D_DIR, 'evaluation')
if os.path.exists(ho3d_eval):
    seqs = os.listdir(ho3d_eval)
    print(f'  Found {len(seqs)} sequences')
    for s in HO3D_SEQUENCES:
        print(f'     [{"OK" if s in seqs else "MISSING"}] {s}')
        if s not in seqs: all_ok = False
else:
    print(f'  Not found: {ho3d_eval}')
    print(f'  Download: gdown --folder 1Wk-HZDvUExyUrRn7us4WWEbHnnFHgOAX -O {HO3D_DIR}/')
    all_ok = False

print('\n[3] YCB Video Models:')
ycb_path = os.path.join(HO3D_DIR, 'models')
print(f'  [{"OK" if os.path.exists(ycb_path) else "MISSING"}] {ycb_path}')
if not os.path.exists(ycb_path): all_ok = False

print('\n[4] Docker + Any6D:')
if not check_docker(ANY6D_DIR):
    all_ok = False

print('\n[5] Docker volume mounts:')
mounts = check_docker_volumes(ANY6D_DIR)
if not mounts.get('evaluation'):
    all_ok = False

print(f'\n{"="*45}')
print('All prerequisites OK' if all_ok else 'Fix missing items above before running Cell 3')

## Cell 3 — Run `run_ho3d_query.py`

Streams output in real time.

**Expected time: ~3-5 hours**

In [ ]:
if not all_ok:
    print('Prerequisites not met — fix Cell 2 first')
else:
    print('Running run_ho3d_query.py in Docker...')

    skip_keywords = [
        'FutureWarning', 'torch.load', 'weights_only', 'ckpt_dir',
        'tensor_type', 'pytree', 'torch.set_default', 'torch.cuda.amp',
        'torch.amp', 'pickle', 'allowlist', 'open an issue'
    ]

    process = subprocess.Popen(
        ['docker', 'compose', 'run', '--rm', '--entrypoint', '', 'any6d', 'bash', '-c',
         'cd /workspace && python run_ho3d_query.py '
         '--anchor_path /workspace/anchor_results/dexycb_reference_view_ours '
         '--hot3d_data_root /workspace/dataset/ho3d '
         '--ycb_model_path /workspace/dataset/ho3d'],
        cwd=ANY6D_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        line = line.rstrip()
        if not any(k in line for k in skip_keywords):
            print(line)

    process.wait()
    subprocess.run(['chmod', '-R', '777', RESULTS_ROOT], capture_output=True)

    if process.returncode == 0:
        print('run_ho3d_query.py completed successfully')
    else:
        print(f'Failed with return code: {process.returncode}')

## Cell 4 — Load Results

In [ ]:
def load_excel(path):
    """Load Excel file using openpyxl. Returns (headers, data rows)."""
    wb   = openpyxl.load_workbook(path)
    ws   = wb.active
    rows = list(ws.values)
    return list(rows[0]), [list(r) for r in rows[1:]]


values       = {}
means        = {}
classes      = []
metrics_cols = []
obj_rows     = []
mean_rows    = []
LATEST_RUN   = None

if not os.path.exists(RESULTS_ROOT):
    print('No results yet — run Cell 3 first')
else:
    name_dir = os.path.join(RESULTS_ROOT, 'any6d')
    runs = sorted(os.listdir(name_dir)) if os.path.exists(name_dir) else []

    if not runs:
        print('No completed runs found')
    else:
        LATEST_RUN   = os.path.join(name_dir, runs[-1])
        global_excel = os.path.join(LATEST_RUN, '0_mean_all_metrics_classes_results.xlsx')
        print(f'Loading from: {LATEST_RUN}\n')

        if os.path.exists(global_excel):
            headers, data = load_excel(global_excel)
            metrics_cols  = headers[1:]
            obj_rows      = [r for r in data if str(r[0]) != 'MEAN']
            mean_rows     = [r for r in data if str(r[0]) == 'MEAN']
            classes       = [str(r[0]) for r in obj_rows]

            for i, m in enumerate(metrics_cols):
                values[m] = [float(r[i+1]) for r in obj_rows]
            if mean_rows:
                for i, m in enumerate(metrics_cols):
                    means[m] = float(mean_rows[0][i+1])

            print('=== HO3D QUERY RESULTS (CVPR 2025) ===')
            print(f'  {"Class":<25}' + ''.join(f'{m:>8}' for m in metrics_cols))
            print(f'  {"-"*75}')
            for row in obj_rows:
                print(f'  {str(row[0]):<25}' + ''.join(f'{float(v):>8.1f}' for v in row[1:]))
            if mean_rows:
                print(f'  {"-"*75}')
                print(f'  {"MEAN":<25}' + ''.join(f'{float(v):>8.1f}' for v in mean_rows[0][1:]))

## Cell 5 — Plot Main Metrics

Bar charts for AR, VSD, MSSD, MSPD and ADD / ADD-S per sequence.

In [ ]:
if not values:
    print('No results loaded — run Cell 4 first')
else:
    # AR / VSD / MSSD / MSPD
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    colors_map = {'AR': '#1565C0', 'VSD': '#2E7D32', 'MSSD': '#E65100', 'MSPD': '#6A1B9A'}

    for ax, metric in zip(axes.flatten(), ['AR', 'VSD', 'MSSD', 'MSPD']):
        if metric not in values:
            ax.text(0.5, 0.5, f'{metric}\nnot available', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue
        vals  = values[metric]
        mean  = means.get(metric, float(np.mean(vals)))
        color = colors_map[metric]
        bars  = ax.bar(range(len(classes)), vals,
                       color=[color if v >= mean else '#FFCCBC' for v in vals],
                       alpha=0.85, width=0.6, edgecolor='white')
        ax.axhline(mean, color='black', linestyle='--', linewidth=2, label=f'Mean = {mean:.1f}%')
        ax.set_xticks(range(len(classes)))
        ax.set_xticklabels(classes, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel(f'{metric} (%)', fontsize=11)
        ax.set_title(metric)
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 105)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=7)

    plt.suptitle('Any6D — HO3D Evaluation Results (CVPR 2025)  |  BOP metrics per sequence')
    plt.tight_layout()
    save_fig(fig, os.path.join(VIZ_DIR, 'ho3d_main_metrics.png'))
    plt.show()

    # ADD / ADD-S
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, metric, color in zip(axes, ['ADD-S', 'ADD'], ['#1976D2', '#388E3C']):
        if metric not in values:
            ax.text(0.5, 0.5, f'{metric}\nnot available', ha='center', va='center', transform=ax.transAxes)
            continue
        vals = values[metric]
        mean = means.get(metric, float(np.mean(vals)))
        bars = ax.bar(range(len(classes)), vals, color=color, alpha=0.85, width=0.6, edgecolor='white')
        ax.axhline(mean, color='black', linestyle='--', linewidth=2, label=f'Mean = {mean:.1f}%')
        ax.set_xticks(range(len(classes)))
        ax.set_xticklabels(classes, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel(f'{metric} (%)', fontsize=11)
        ax.set_title(metric)
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 105)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8)

    plt.suptitle('Any6D — HO3D Results  |  ADD / ADD-S Metrics')
    plt.tight_layout()
    save_fig(fig, os.path.join(VIZ_DIR, 'ho3d_add_metrics.png'))
    plt.show()

## Cell 6 — Results Table

In [ ]:
if not values:
    print('No results loaded — run Cell 4 first')
else:
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.axis('off')

    col_labels = ['Sequence'] + metrics_cols
    table_data = []
    for row in obj_rows:
        table_data.append([str(row[0])] + [f'{float(v):.1f}' for v in row[1:]])
    if mean_rows:
        table_data.append(['MEAN'] + [f'{float(v):.1f}' for v in mean_rows[0][1:]])

    table = ax.table(cellText=table_data, colLabels=col_labels, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 2.0)

    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#1565C0')
        table[0, j].set_text_props(color='white', fontweight='bold')

    for i in range(1, len(table_data) + 1):
        for j in range(len(col_labels)):
            if table_data[i-1][0] == 'MEAN':
                table[i, j].set_facecolor('#E3F2FD')
                table[i, j].set_text_props(fontweight='bold')
            elif i % 2 == 0:
                table[i, j].set_facecolor('#F5F5F5')

    plt.title('Any6D — HO3D Evaluation Table (CVPR 2025)  |  Metrics: ADD-S, ADD, AR, MSSD, MSPD, VSD (%)', pad=20)
    save_fig(fig, os.path.join(VIZ_DIR, 'ho3d_results_table.png'))
    plt.show()

## Cell 7 — Final Summary

In [ ]:
print('=' * 55)
print('HO3D QUERY PIPELINE — FINAL SUMMARY')
print('=' * 55)

if not os.path.exists(RESULTS_ROOT):
    print('\n  No results yet — run Cell 3 first')
    print(f'\n  To download HO3D dataset:')
    print(f'  nohup gdown --folder 1Wk-HZDvUExyUrRn7us4WWEbHnnFHgOAX \\')
    print(f'        -O {HO3D_DIR}/ > ~/download_ho3d.log 2>&1 &')
else:
    if LATEST_RUN:
        print(f'\n  Results: {LATEST_RUN}')

    if means:
        print('\n  Mean metrics (all sequences):')
        for metric, val in means.items():
            print(f'  {metric:<8} {float(val):>6.1f}%')

    print('\n  Output files:')
    for f in ['ho3d_main_metrics.png', 'ho3d_add_metrics.png', 'ho3d_results_table.png']:
        path   = os.path.join(VIZ_DIR, f)
        status = 'OK' if os.path.exists(path) else 'MISSING'
        print(f'  [{status}] {f}')